# Statistics of FORGE-Solana Dataset

In [1]:
import os
import json
from dataclasses import dataclass
from dataclasses import field
from typing import List, Dict, Union, Optional, Literal
from pydantic import BaseModel, RootModel, Field
from collections import namedtuple,Counter

@dataclass
class ProjectInfo:
    url: Union[str, int, List, None] = "n/a"
    commit_id: Union[str, int, List, None] = "n/a"
    address: Union[str, int, List, None] = "n/a"
    chain: Union[str, int, List, None] = "n/a"
    compiler_version: Union[str, List, None] = "n/a"
    audit_date: Union[str, int, List, None] = "n/a"
    project_path: Union[str, List, Dict, None] = "n/a"

    def is_empty(self):
        if (self.url == "n/a" and self.address == "n/a") or (
            not self.url and not self.address
        ):
            return True
        return False

@dataclass
class Finding:
    id: Union[str, int] = 0
    category: Dict = field(default_factory=dict)
    title: str = ""
    description: str = ""
    severity: Optional[str] = ""
    location: Union[str, int, List] = ""


class Report(BaseModel):
    path: str = ""
    project_info: ProjectInfo = field(default_factory=ProjectInfo)
    findings: List[Finding] = field(default_factory=list)

    def append_finding(self, finding: Finding):
        self.findings.append(finding)


class JSONFileProcessor:
    def __init__(self, directory: str):
        self.directory = directory
        self.file_count = 0

    def _get_all_json_files(self) -> List[str]:
        json_files = []
        for root, _, files in os.walk(self.directory):
            for file in files:
                if file.endswith(".json"):
                    json_files.append(os.path.join(root, file))
                    self.file_count += 1
        return json_files

    def operate_add(self,results:List,result_type):
        res = {}
        for field in result_type._fields:
            if isinstance(getattr(results[0],field),int):
                res[field] = 0
            else:
                res[field] = []
            # res[field] = 0
        for result in results:
            for field in result._fields:
                res[field] += getattr(result, field)

        return res
    def operate_reduce(self,results:List,result_type):
        res = {}
        for field in result_type._fields:
            res[field] = []
        for result in results:
            for field in result._fields:
                res[field].extend(getattr(result, field))
        return res

    def process_files(
        self, analysis_func=None, modify_func=None
    ) -> List[Report]:
        results = []
        json_files = self._get_all_json_files()
        for json_file in json_files:
            # print(f"Processing file: {json_file}")
            with open(json_file, "r", encoding="utf8") as f:
                data = json.load(f)
            report = Report(**data)
            if analysis_func:
                result = analysis_func(report)
                results.append(result)
            if modify_func:
                modified_report: Report = modify_func(report)
                with open(json_file, "w", encoding="utf8") as f:
                    f.write(modified_report.model_dump_json(indent=4,exclude={'project_info': {'compiler_version'}}))
        return results


## Count findings & projects

In [2]:
Finding = namedtuple("Finding", ["total_files","total_projects","total_findings"])

def count_finding(report: Report):
    if isinstance(report.project_info.url,list):
        # print(report.path)
        project_count = len(report.project_info.url)
    else:
        project_count = 1
    result = Finding(total_findings=len(report.findings),total_files=1,total_projects=project_count)
    return result

processor = JSONFileProcessor("../../dataset/findings")
results = processor.process_files(analysis_func=count_finding)
res = processor.operate_add(results,Finding)
res

{'total_files': 199, 'total_projects': 217, 'total_findings': 1582}

## Check & count contracts

In [3]:
import re
from pathlib import Path

Result = namedtuple(
    "Result",
    [
        "total_files",
        "total_projects", 
        "valid_projects",
        "rust_files",
        "lines_of_code",
    ],
)

def check_projects(report: Report):
    def count_lines(filepath: Path):
        try:
            with open(filepath, "r", encoding="utf8") as rust_file:
                lines = len(rust_file.readlines())
                return lines
        except Exception as e:
            print(f"Error opening file {filepath}: {e}")
            return 0  # Bug fix: should return 0, not tuple
    
    if report.project_info.is_empty():
        valid_projects = 0
        rust_files = 0
        lines_of_code = 0
        total_projects = 0  # Bug fix: handle empty case properly
        print(f"Empty project info in {report.path}")
    else:
        valid_projects = len(report.project_info.url) if isinstance(report.project_info.url, list) else 1
        rust_files = 0
        lines_of_code = 0
        
        # Bug fix: handle project_path properly
        v = report.project_info.project_path
        if v:
            total_projects = len(report.project_info.url) if isinstance(report.project_info.url, list) else 1
            project_path = Path("../../dataset") / v.lstrip('/')
            if project_path.exists():
                rust_file_paths = [p for p in project_path.glob("**/*.rs") if p.is_file()]
                rust_files += len(rust_file_paths)
                for path in rust_file_paths:
                    lines_of_code += count_lines(path)
        else:
            total_projects = 0
    
    return Result(
        total_files=1,
        total_projects=total_projects,
        valid_projects=valid_projects,
        rust_files=rust_files,
        lines_of_code=lines_of_code,
    )

processor = JSONFileProcessor("../../dataset/findings")
results = processor.process_files(analysis_func=check_projects)
res = processor.operate_add(results, Result)

res

{'total_files': 199,
 'total_projects': 217,
 'valid_projects': 217,
 'rust_files': 21047,
 'lines_of_code': 5762695}

In [ ]:
average_lines_of_code = res["lines_of_code"] / res["valid_projects"]
average_files_per_project = res["rust_files"] / res["valid_projects"]
print(f"Total files: {res['total_files']}")
print(f"Total projects: {res['total_projects']}")
print(f"Valid projects: {res['valid_projects']}")
print(f"Total rust files: {res['rust_files']}")
print(f"Total lines of code: {res['lines_of_code']}")
print(f"Average lines of code: {average_lines_of_code}")
print(f"Average files per project: {average_files_per_project}")

Total files: 199
Total projects: 217
Valid projects: 217
Total rust files: 21047
Total lines of code: 5762695
Average lines of code: 26556.19815668203
Average files per project: 96.99078341013825


## Severity statistics

In [5]:
severity_category = ["na","informational","low","medium","high","critical"]

Severity = namedtuple("Category",severity_category)
def count_severity(report: Report):
    findings = report.findings
    severity_dict = {"na": 0, "informational": 0, "low": 0, "medium": 0, "high": 0, "critical": 0}
    for finding in findings:
        severity = finding.severity
        if severity:
            severity = finding.severity.lower().strip()
        
        if severity == None or severity == "" or severity == "n/a":
            severity = "na"
        
        
        if severity not in severity_category:
            print(f"Unknown severity: {severity}")
            continue
        severity_dict[severity] = severity_dict.get(severity, 0) + 1
        # dict to namedtuple
    return Severity(**severity_dict)

processor = JSONFileProcessor("../../dataset/findings")
results = processor.process_files(analysis_func=count_severity)
res = processor.operate_add(results, Severity)
res

{'na': 50,
 'informational': 464,
 'low': 454,
 'medium': 346,
 'high': 211,
 'critical': 57}